In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import os
import pickle

from sklearn.preprocessing import StandardScaler
from category_encoders import TargetEncoder

# -------------------- Load and Merge Test Datasets --------------------
test_transaction = pd.read_csv("datasets/test_transaction.csv")
test_identity = pd.read_csv("datasets/test_identity.csv")

# Normalize test_identity column names to match train_identity
test_transaction.columns = test_transaction.columns.str.replace('-', '_').str.strip()
test_identity.columns = test_identity.columns.str.replace('-', '_').str.strip()

# Merge on 'TransactionID'
test_data = test_transaction.merge(test_identity, how="left", on="TransactionID")

# Store TransactionID for submission
submission = pd.DataFrame()
submission["TransactionID"] = test_data["TransactionID"]

# -------------------- Preprocessing --------------------

# Handle missing values
categorical_cols = test_data.select_dtypes(include="object").columns.tolist()
numerical_cols = test_data.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Fill missing values: categorical → "Missing", numerical → median
test_data[categorical_cols] = test_data[categorical_cols].fillna("Missing")
for col in numerical_cols:
    median_val = test_data[col].median()
    test_data[col] = test_data[col].fillna(median_val)

# -------------------- Load Training Data and Splits --------------------
train_data = pd.read_csv("datasets/train_dataset.csv")
X_full = train_data.drop(columns=["isFraud"])
y_full = train_data["isFraud"]

with open("10_fold_indices.pkl", "rb") as f:
    splits = pickle.load(f)

# Load selected features (95% cumulative importance set)
selected_features_df = pd.read_csv("feature_importance_reduced/lgbm_feature_importance_target_encoded_2.csv")
selected_features = selected_features_df["feature"].tolist()
X_test = test_data.copy()

# -------------------- Target Encoding and Scaling --------------------
# Fit TargetEncoder on full training data
categorical_cols = X_full.select_dtypes(include="object").columns.tolist()
numerical_cols = X_full.select_dtypes(include=["int64", "float64"]).columns.tolist()

encoder = TargetEncoder(cols=categorical_cols)
X_full[categorical_cols] = encoder.fit_transform(X_full[categorical_cols], y_full)
X_test[categorical_cols] = encoder.transform(X_test[categorical_cols])

# Standard scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_full[numerical_cols] = scaler.fit_transform(X_full[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

# Keep only selected features
X_test = X_test[selected_features]

# -------------------- Load Models and Predict --------------------
predictions = np.zeros(len(X_test))

for fold in range(1, 11):
    print(f"Loading model for Fold {fold}...")
    model_path = f"models/lightgbm_reduced/fold_{fold}_2.txt"
    model = lgb.Booster(model_file=model_path)

    fold_preds = model.predict(X_test, num_iteration=model.best_iteration)
    predictions += fold_preds

# Average predictions from all folds
predictions /= 10

# -------------------- Generate Submission --------------------
submission["isFraud"] = predictions
submission.to_csv("IEEE-CIS_Submission/lightgbm-submission_86-dataset.csv", index=False)
print("Submission file 'submission_86_dataset.csv' created successfully!")


Loading model for Fold 1...
Loading model for Fold 2...
Loading model for Fold 3...
Loading model for Fold 4...
Loading model for Fold 5...
Loading model for Fold 6...
Loading model for Fold 7...
Loading model for Fold 8...
Loading model for Fold 9...
Loading model for Fold 10...
Submission file 'submission_86_dataset.csv' created successfully!
